# Introduction to Euler, Runge, and Kutta

## Leonhard Euler (1707–1783)
[Leonhard Euler](https://en.wikipedia.org/wiki/Leonhard_Euler) was a Swiss mathematician whose contributions laid the foundations for much of modern calculus and analysis. What is now called the [Euler method](https://en.wikipedia.org/wiki/Euler_method) was one of the first attempts to numerically solve ordinary differential equations (ODEs). It was developed to address mechanical problems, such as predicting the motion of planets or pendulums.

## Carl Runge (1856–1927) and Martin Wilhelm Kutta (1867–1944)
[Carl Runge](https://en.wikipedia.org/wiki/Carl_Runge) and [Martin Wilhelm Kutta](https://en.wikipedia.org/wiki/Martin_Kutta) were German mathematicians who improved Euler's method to create more accurate numerical solutions. (Runge was also a physicist and spectroscopist.) Kutta was trying to solve for atomic spectra, while Runge was interested in an aerofoil. The Runge-Kutta methods, including the RK2 and RK4 methods, use intermediate slope calculations to reduce error, making them essential tools for solving complex, [stiff differential equations](https://en.wikipedia.org/wiki/Stiff_equation).

# Solving the Differential Equation: $\dot{y} = \gamma y$

This equation describes exponential growth or decay, depending on the value of $\gamma$. Its analytic solution is:

$$
y(t) = y_0 e^{\gamma t}
$$

In this section, we'll solve this numerically using the Euler method and the RK2 method. We can use plotting and widgets to compare the numerical results to the analytic solution. We will also explore the effects of different step sizes. **This approach of testing our code against an analytic result is an important part of coding.** Once we have tested it on things we know, we can apply it to things we don't know.


# Euler's Method Setup:

The Euler method is a simple yet way to approximate the solution of ordinary differential equations (ODEs). It steps forward in time, updating the solution with the derivative at the current point.

Take the following first-order ODE:

$$
\frac{dy}{dt} = f(t, y)
$$

Euler's method approximates the solution at discrete time steps. Starting from an initial value $y_0 = y(t_0)$, we can approximate the value of $y$ at the follwing time steps by using the formula:

$$
y_{n+1} = y_n + f(t_n, y_n) \cdot \Delta t
$$

Here,
- $y_n$ is the approximation of $y(t)$ at the $n$-th time step;
- $\Delta t$ is the time step;
- $f(t_n, y_n)$ is the derivative of $y$ at time $t_n$.

### Discrete Setup:
In Python, we can store the values of $y$ and $t$ as arrays or lists. Here's how we set it up:

1. **Initial Conditions**: 
   - The initial value of $y$ is stored as $y[0]$.
   - The initial time is $t[0] = 0$ (or some other starting time).

2. **Time Array**: 
   - We define the time steps using `np.arange`, which generates an array of evenly spaced time points from $t_0$ to $t_{\text{end}}$.

3. **Euler Update**:
   - For each time step, we compute the next value of $y$ using the equation $y_{n+1} = y_n + f(t_n, y_n) \cdot \Delta t$.
   - We repeat this for all time steps, updating $y$ based on the current value of $y$ and the derivative $f(t_n, y_n)$.

### Example:
For the differential equation:

$$
\dot{y} = \gamma y
$$

Euler's method takes the form:

$$
y_{n+1} = y_n + \gamma y_n \cdot \Delta t
$$

Where:
- $y_n$ is the current value of $y$,
- $\gamma$ is a constant (either positive or negative, depending on whether it's exponential growth or decay),
- $\Delta t$ is the time step.

We will implement this approach step-by-step in the next code cell, where $y[0]$ will be the initial value $y_0$, and we will update it iteratively for all time steps.


## Question 1: Exploring the Euler method

1. Use the equation $y_{n+1} = y_n + \gamma y_n \cdot \Delta t$ to write out a few terms, as Euler would have, and compare with the plot below.
2. Read through the following code and make sure you understand the steps.
3. Add comments to the code, for your future self.
4. Explore the various sliders.
5. If you like, add plot points, change the linewidth, and plot range, if needed. (e.g. `plot(x, y, color='green', marker='.' linewidth=1, markersize=12)`)
6. Similarly, if you think it would be helpful, try using `plt.semilogy` instead of `plt.plot` in `plot_solution`;


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

def euler_method(gamma, y0, t_end, dt): #euler method definition 
    t_vals = np.arange(0, t_end, dt) 
    y_vals = np.zeros(len(t_vals))
    y_vals[0] = y0 
    for i in range(1, len(t_vals)):
        y_vals[i] = y_vals[i-1] + gamma * y_vals[i-1] * dt
    return t_vals, y_vals

def analytic_solution(gamma, y0, t_vals): #analytic definition
    return y0 * np.exp(gamma * t_vals)

def plot_solution(gamma=1, y0=1, t_end=10, dt=0.1): #plot solution
    t_vals, euler_vals = euler_method(gamma, y0, t_end, dt)
    analytic_vals = analytic_solution(gamma, y0, t_vals)
    
    plt.plot(t_vals, euler_vals, label="Euler Method") #creating the plot 
    plt.plot(t_vals, analytic_vals, label="Analytic Solution", linestyle='dashed')
    plt.xlabel('Time')
    plt.ylabel('y(t)')
    plt.legend()
    plt.title(f"Comparison of Euler and Analytic Solutions (dt={dt})")
    plt.show()

interact(plot_solution, 
         gamma=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$\gamma$'), 
         y0=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$y(0)$'), 
         t_end=FloatSlider(value=2.0, min=1.0, max=20.0, step=0.1, description='$t_{end}$'), 
         dt=FloatSlider(value=1.0, min=0.1, max=1.0, step=0.1, description='$dt$')
        );


interactive(children=(FloatSlider(value=1.0, description='$\\gamma$', max=10.0, min=-10.0), FloatSlider(value=…

# Runge-Kutta 2 (RK2) Method for $\dot{y} = \gamma y$

The RK2 method uses the Euler method to find one slope, then uses the result to approximate an intermediate slope. We'll take a slightly closer look later.

Here's is the procedure:

1. Compute the first slope: $k_1 = \dot{y}(t_n, y_n)$
2. Compute the second slope: $k_2 = \dot{y}(t_n + \frac{dt}{2}, y_n + \frac{dt}{2} k_1)$
3. Estimate the next value: $y_{n+1} = y_n + k_2 \cdot dt$


## Question 2: Exploring the RK2 method

1. Read through the following code and make sure you understand the steps.
2. Add comments to the code, for your future self.
3. Explore the various sliders.
4. Add plot points, change the linewidth, and plot range, if needed. (e.g. `plot(x, y, color='green', marker='.' linewidth=1, markersize=12)`)
5. Try using `plt.semilogy` instead of `plt.plot` in `plot_solution`;


In [2]:
def rk2_method(gamma, y0, t_end, dt): #first method uses some type of range of values
    t_vals = np.arange(0, t_end, dt)
    y_vals = np.zeros(len(t_vals))
    y_vals[0] = y0
    for i in range(1, len(t_vals)):
        k1 = gamma * y_vals[i-1]
        k2 = gamma * (y_vals[i-1] + dt * k1 / 2)
        y_vals[i] = y_vals[i-1] + k2 * dt
    return t_vals, y_vals

def plot_rk2_solution(gamma=1, y0=1, t_end=10, dt=0.1): #creating a plot for the method
    t_vals, euler_vals = euler_method(gamma, y0, t_end, dt)
    t_vals, rk2_vals = rk2_method(gamma, y0, t_end, dt)
    analytic_vals = analytic_solution(gamma, y0, t_vals)

    #plt.plot(t_vals, euler_vals, label="Euler Method")
    #plt.plot(t_vals, rk2_vals, label="RK2 Method")
    #plt.plot(t_vals, analytic_vals, label="Analytic Solution", linestyle='dashed')
    plt.semilogy(t_vals, euler_vals, label="Euler Method") #changes plot from linear to log for a cleaner picture of what is changing
    plt.semilogy(t_vals, rk2_vals, label="RK2 Method")
    plt.semilogy(t_vals, analytic_vals, label="Analytic Solution", linestyle='dashed')
    plt.xlabel('Time')
    plt.ylabel('y(t)')
    plt.legend()
    plt.title(f"Comparison of Euler, RK2, and Analytic Solutions (dt={dt})")
    plt.show()

interact(plot_rk2_solution, 
         gamma=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$\gamma$'), 
         y0=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$y(0)$'), 
         t_end=FloatSlider(value=2.0, min=1.0, max=20.0, step=0.1, description='$t_{end}$'), 
         dt=FloatSlider(value=1.0, min=0.1, max=1.0, step=0.1, description='$dt$')
        );


interactive(children=(FloatSlider(value=1.0, description='$\\gamma$', max=10.0, min=-10.0), FloatSlider(value=…

# Solving the harmonic oscillator: $m\ddot{y} + k y = 0$

This equation describes the motion of a **harmonic oscillator**. 

The second-order differential equation is:

$$
m\ddot{y} + k y = 0
$$

Where:
- $m$ is the mass of the object,
- $k$ is the spring constant,
- $y(t)$ is the displacement from the equilibrium position at time $t$,
- $\ddot{y}(t)$ is the acceleration (second derivative of $y$).

### Step 1: Convert the Second-Order ODE to a System of First-Order ODEs

To solve this equation numerically using the **Euler method**, we need to reduce it to a system of **first-order** differential equations. This approach is standard for setting up numerical solutions to differential equations. It is also used in many analytic approaches. We do this by introducing a new variable for the velocity:

Let:
- $v = \dot{y}$ represent the velocity.

Now we have a system of two first-order equations:

1. The first equation is simply the definition of velocity:
   $$
   \dot{y} = v
   $$

2. The second equation comes from the original ODE. Using $ \dot{v} = \ddot{y} $, we get:
   $$
   \dot{v} = -\frac{k}{m} y
   $$

Thus, we have the following system of first-order ODEs:
$$
\begin{aligned}
\dot{y} &= v \\
\dot{v} &= -\frac{k}{m} y
\end{aligned}
$$

This system can now be solved using numerical methods like Euler's method.

### Step 2: Discretizing the System Using Euler’s Method

We can now apply Euler's method to approximate the solution. For each time step, we update both $y$ and $v$ using their respective differential equations:

$$
y_{n+1} = y_n + \dot{y}_n \cdot \Delta t = y_n + v_n \cdot \Delta t
$$

$$
v_{n+1} = v_n + \dot{v}_n \cdot \Delta t = v_n - \frac{k}{m} y_n \cdot \Delta t
$$

Where:
- $y_n$ is the displacement at time step $n$,
- $v_n$ is the velocity at time step $n$,
- $\Delta t$ is the size of the time step.

### Step 3: Initial Conditions

To solve the system numerically, we also need initial conditions. These include:
- $y(0)$, the initial displacement,
- $v(0)$, the initial velocity.

Given the initial conditions, we can use the Euler method to iteratively update $y$ and $v$ at each time step, predicting their future values.

### Step 4: Implementation of Euler's Method

We will now implement the Euler method for this system of equations. Starting from the initial conditions $y(0)$ and $v(0)$, we will compute the values of $y$ and $v$ at future time steps, simulating the oscillatory motion of the system.

Let's see what happens.

## Question 3: Exploring the Euler method applied to a second order equation

1. Make sure you can derive the equation of motion from the potential energy.
2. Read through the following code and make sure you understand the steps.
3. Add comments to the code, for your future self.
4. Add an analytic solution to compare with.
5. Explore the various sliders.

In [3]:
def euler_second_order(m, k, y0, v0, t_end, dt):
    t_vals = np.arange(0, t_end, dt)
    y_vals = np.zeros(len(t_vals))
    v_vals = np.zeros(len(t_vals))
    y_vals[0] = y0
    v_vals[0] = v0
    
    for i in range(1, len(t_vals)):
        v_vals[i] = v_vals[i-1] - (k/m) * y_vals[i-1] * dt
        y_vals[i] = y_vals[i-1] + v_vals[i-1] * dt
    
    return t_vals, y_vals, v_vals

def analytic_solution(m, k, y0, v0, t_vals):
    omega = np.sqrt(k / m)
    y_analytic = y0 * np.cos(omega * t_vals) + (v0 / omega) * np.sin(omega * t_vals)
    return y_analytic

def plot_second_order(m=1, k=1, y0=1, v0=0, t_end=10, dt=0.1):
    t_vals, y_vals, v_vals = euler_second_order(m, k, y0, v0, t_end, dt)
    y_analytic = analytic_solution(m, k, y0, v0, t_vals)
    plt.plot(t_vals, y_analytic, label="y(t) Analytic (Position)", linestyle='dashed')
    plt.plot(t_vals, y_vals, label="y(t) (Position)")
    plt.plot(t_vals, v_vals, label="v(t) (Velocity)")
    plt.xlabel('Time')
    plt.ylabel('Values')
    plt.legend()
    plt.title(f"Solution to $m\\ddot{{y}} + ky = 0$ (dt={dt})")
    plt.show()

interact(plot_second_order,
         m=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='$m$'),  
         k=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='$k$'),
         y0=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$y(0)$'), 
         v0=FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description='$\dot{y}(0)$'), 
         t_end=FloatSlider(value=2.0, min=1.0, max=20.0, step=0.1, description='$t_{end}$'), 
         dt=FloatSlider(value=1.0, min=0.1, max=1.0, step=0.1, description='$dt$')
        );


interactive(children=(FloatSlider(value=1.0, description='$m$', max=10.0, min=0.1), FloatSlider(value=1.0, des…

# Solving the Harmonic Oscillator with the RK2 Method

Let's see what RK2 can do. We start with the differential equation:

$$
m\ddot{y} + k y = 0
$$

we've already reduced it to the system of first-order equations:

$$
\dot{y} = v
$$
$$
\dot{v} = -\frac{k}{m} y
$$

### RK2 Algorithm
For each step, the RK2 method estimates the next values $y_{n+1}$ and $v_{n+1}$ by performing two evaluations of the derivative at intermediate points. The steps are:

1. Compute the first set of slopes:
   $$
   k_1^y = v_n
   $$
   $$
   k_1^v = -\frac{k}{m} y_n
   $$

2. Use these slopes to compute the intermediate estimates of $y$ and $v$:
   $$
   \tilde{y} = y_n + k_1^y \frac{\Delta t}{2}
   $$
   $$
   \tilde{v} = v_n + k_1^v \frac{\Delta t}{2}
   $$

3. Compute the second set of slopes based on the intermediate estimates:
   $$
   k_2^y = \tilde{v}
   $$
   $$
   k_2^v = -\frac{k}{m} \tilde{y}
   $$

4. Finally, update $y$ and $v$ using these second slopes:
   $$
   y_{n+1} = y_n + k_2^y \cdot \Delta t
   $$
   $$
   v_{n+1} = v_n + k_2^v \cdot \Delta t
   $$

This turns out to be more complicated to write than to code.

## Question 4: Exploring the RK2 method applied to a the harmonic oscillator

2. Read through the following code and make sure you understand the steps.
3. Add comments to the code, for your future self.
4. Add an analytic solution to compare with.
5. Explore the various sliders.

In [4]:
def rk2_second_order(m, k, y0, v0, t_end, dt): #RK 2 method
    t_vals = np.arange(0, t_end, dt)
    y_vals = np.zeros(len(t_vals))
    v_vals = np.zeros(len(t_vals))
    y_vals[0] = y0
    v_vals[0] = v0
    
    for i in range(1, len(t_vals)):
        # First slopes (k1)
        k1_y = v_vals[i-1]
        k1_v = -(k/m) * y_vals[i-1]
        
        # Intermediate estimates
        y_mid = y_vals[i-1] + k1_y * dt / 2
        v_mid = v_vals[i-1] + k1_v * dt / 2
        
        # Second slopes (k2)
        k2_y = v_mid
        k2_v = -(k/m) * y_mid
        
        # Update the values
        y_vals[i] = y_vals[i-1] + k2_y * dt
        v_vals[i] = v_vals[i-1] + k2_v * dt
    
    return t_vals, y_vals, v_vals

def euler_second_order(m, k, y0, v0, t_end, dt):
    t_vals = np.arange(0, t_end, dt)
    y_vals = np.zeros(len(t_vals))
    v_vals = np.zeros(len(t_vals))
    y_vals[0] = y0
    v_vals[0] = v0
    
    for i in range(1, len(t_vals)):
        v_vals[i] = v_vals[i-1] - (k/m) * y_vals[i-1] * dt
        y_vals[i] = y_vals[i-1] + v_vals[i-1] * dt
    
    return t_vals, y_vals, v_vals

def plot_rk2_second_order(m=1, k=1, y0=1, v0=0, t_end=10, dt=0.1):
    t_vals, euler_y_vals, euler_v_vals = euler_second_order(m, k, y0, v0, t_end, dt)
    t_vals, rk2_y_vals, rk2_v_vals = rk2_second_order(m, k, y0, v0, t_end, dt)
    
    plt.plot(t_vals, euler_y_vals, label="Euler Method - Position")
    plt.plot(t_vals, rk2_y_vals, label="RK2 Method - Position")
    plt.plot(t_vals, euler_v_vals, label="Euler Method - Velocity", linestyle="dashed")
    plt.plot(t_vals, rk2_v_vals, label="RK2 Method - Velocity", linestyle="dashed")
    
    plt.xlabel('Time')
    plt.ylabel('y(t), v(t)')
    plt.legend()
    plt.title(f"Comparison of Euler and RK2 for Harmonic Oscillator (dt={dt})")
    plt.show()

interact(plot_rk2_second_order,
         m=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='$m$'),  
         k=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='$k$'),
         y0=FloatSlider(value=1.0, min=-10.0, max=10.0, step=0.1, description='$y(0)$'), 
         v0=FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description='$\\dot{y}(0)$'), 
         t_end=FloatSlider(value=2.0, min=1.0, max=20.0, step=0.1, description='$t_{end}$'), 
         dt=FloatSlider(value=1.0, min=0.01, max=1.0, step=0.01, description='$dt$')
        );


interactive(children=(FloatSlider(value=1.0, description='$m$', max=10.0, min=0.1), FloatSlider(value=1.0, des…